# Phyloseq-working

## 10.17.26 | tidy metadata

### Setting up R/phyloseq with conda environment in unity

Add phyloseq to `dada2-env` in terminal
```
conda activate dada2-env
conda install -c bionconda bioconda-phyloseq
```

### Metadata tidying

In [1]:
# Load packages
suppressPackageStartupMessages({
    library(phyloseq)
    library(Biostrings)
    library(tidyverse)
})

In [32]:
# Read 16S processing tracker csv
meta16S <- read.csv("/project/pi_sarah_gignouxwolfsohn_uml_edu/jade/ORCC_16S/metadata/16S_tracker.csv")
dim(meta16S)
head(meta16S)

[1] 283  30

,Original_order,Sample_name,Phase,Tissue,Concentration,DNA_dilution,X16S_date,X16S_tube_number,X16S_adapters,X16S_tray,⋯,Pooled_notes,Index_primers,Index_date,Index_volume,Clean_date,Clean_date_2,Gel_extraction,Final_concentration,X25ng,Notes
,<int>,<chr>,<int>,<chr>,<dbl>,<dbl>,<chr>,<int>,<chr>,<int>,⋯,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>
1,2,2024_Both05_Warm03_O33,2,gut,53.0,0.1,12/15/25,2,3N,1,⋯,<15uL,BFi-14,6/26/26,2,6/30/26,7/1/26,NA,6.10,4.10,
2,4,2024_Both06_Hyp06_O72,2,gut,53.8,0.1,12/15/25,4,3N,1,⋯,,BFi-14,6/26/26,2,6/30/26,7/1/26,NA,8.88,2.82,
3,8,2024_Cont02_Warm02_W15,2,gut,29.2,0.1,12/15/25,8,3N,1,⋯,,BFi-14,6/26/26,2,6/30/26,7/1/26,NA,6.64,3.77,
4,10,2024_Both03_Warm01_O6,2,gut,77.8,0.1,12/16/25,2,3N,1,⋯,<15uL,BFi-14,6/26/26,2,6/30/26,7/1/26,NA,10.00,2.50,
5,12,2024_Both03_Warm04_O41,2,gut,86.9,0.1,12/16/25,4,3N,1,⋯,,BFi-14,6/26/26,2,6/30/26,7/1/26,NA,9.86,2.54,
6,14,2024_Both04_Warm02_O19,2,gut,38.7,0.1,12/16/25,6,3N,1,⋯,,BFi-14,6/26/26,2,6/30/26,7/1/26,NA,7.52,3.32,


In [33]:
# Tidy data
meta16S <- meta16S %>% 
                mutate(Original_order = NULL) %>%
                # Un-concatenate sample name
                separate(Sample_name,  into = c("Year", "Phase_1_treat", "Phase_2_treat", "Tag"), sep = "_", remove = FALSE) %>%
                # Add tissue type to sample name to match ASV table
                mutate(Sample_name = paste0(Sample_name, sep = "_", Tissue)) %>%
                # Separate the treatment and replicate into two columns
                extract(Phase_1_treat, into = c("Phase_1_treat", "Phase_1_rep"), regex = "([A-Za-z]+)([0-9]+)") %>%
                extract(Phase_2_treat, into = c("Phase_2_treat", "Phase_2_rep"), regex = "([A-Za-z]+)([0-9]+)") %>%
                # Add column for treatment combination
                mutate(Combined_treat = paste0(Phase_1_treat, Phase_2_treat)) %>%
                mutate(Combined_treat = gsub("[^A-Z]", "", Combined_treat)) %>%
                relocate(Combined_treat, .after = Year) %>%
                # Make the treatment replicate an integer rather than a string
                mutate(Phase_1_rep = as.integer(Phase_1_rep)) %>%
                mutate(Phase_2_rep = as.integer(Phase_2_rep)) %>%
                # Rename some columns
                rename(Date = X16S_date) %>%
                rename(Adapters = X16S_adapters) %>%
                # Format dates
                mutate(Date = as.Date(Date, format = "%m/%d/%y")) %>%
                mutate(Index_date = as.Date(Index_date, format = "%m/%d/%y"))
head(meta16S)

Warning message:
“Expected 4 pieces. Missing pieces filled with `NA` in 7 rows [239, 240, 243,
247, 280, 282, 283].”


,Sample_name,Year,Combined_treat,Phase_1_treat,Phase_1_rep,Phase_2_treat,Phase_2_rep,Tag,Phase,Tissue,⋯,Pooled_notes,Index_primers,Index_date,Index_volume,Clean_date,Clean_date_2,Gel_extraction,Final_concentration,X25ng,Notes
,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<int>,<chr>,<int>,<chr>,⋯,<chr>,<chr>,<date>,<int>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>
1,2024_Both05_Warm03_O33_gut,2024,BW,Both,5,Warm,3,O33,2,gut,⋯,<15uL,BFi-14,2026-06-26,2,6/30/26,7/1/26,NA,6.10,4.10,
2,2024_Both06_Hyp06_O72_gut,2024,BH,Both,6,Hyp,6,O72,2,gut,⋯,,BFi-14,2026-06-26,2,6/30/26,7/1/26,NA,8.88,2.82,
3,2024_Cont02_Warm02_W15_gut,2024,CW,Cont,2,Warm,2,W15,2,gut,⋯,,BFi-14,2026-06-26,2,6/30/26,7/1/26,NA,6.64,3.77,
4,2024_Both03_Warm01_O6_gut,2024,BW,Both,3,Warm,1,O6,2,gut,⋯,<15uL,BFi-14,2026-06-26,2,6/30/26,7/1/26,NA,10.00,2.50,
5,2024_Both03_Warm04_O41_gut,2024,BW,Both,3,Warm,4,O41,2,gut,⋯,,BFi-14,2026-06-26,2,6/30/26,7/1/26,NA,9.86,2.54,
6,2024_Both04_Warm02_O19_gut,2024,BW,Both,4,Warm,2,O19,2,gut,⋯,,BFi-14,2026-06-26,2,6/30/26,7/1/26,NA,7.52,3.32,


In [34]:
# Check warning message
meta16S[239:247,]
meta16S[280:283,]
# Missing pieces are control samples

,Sample_name,Year,Combined_treat,Phase_1_treat,Phase_1_rep,Phase_2_treat,Phase_2_rep,Tag,Phase,Tissue,⋯,Pooled_notes,Index_primers,Index_date,Index_volume,Clean_date,Clean_date_2,Gel_extraction,Final_concentration,X25ng,Notes
,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<int>,<chr>,<int>,<chr>,⋯,<chr>,<chr>,<date>,<int>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>
239,Pooled_negative_rack1_NA,Pooled,NA,NA,NA,rack,1,NA,NA,NA,⋯,,BFi-14,2026-06-26,1,6/30/26,7/1/26,NA,1.14,3.00,Rack 1 negative
240,Pooled_negative_rack2_NA,Pooled,NA,NA,NA,rack,2,NA,NA,NA,⋯,,BFi-15,2026-07-03,1,7/3/26,NA,NA,5.02,3.00,Rack 2 negative; primer dimer that didn't get cleaned
241,2024_Hyp01_Null_B08_gill,2024,HNA,Hyp,1,NA,NA,B08,1,gill,⋯,,BFi-13,2026-07-06,1,7/6/26,NA,NA,11.07,2.26,Diluted final sample 1:3
242,2024_Hyp05_Null_B35_gill,2024,HNA,Hyp,5,NA,NA,B35,1,gill,⋯,,BFi-13,2026-07-06,1,7/6/26,NA,NA,7.53,3.32,Diluted final sample 1:3
243,Pooled_negative_rack3_NA,Pooled,NA,NA,NA,rack,3,NA,NA,NA,⋯,,BFi-13,2026-07-06,1,7/6/26,NA,NA,1.25,3.00,Rack 3 negative
244,2024_Warm02_Null_G15_gill,2024,WNA,Warm,2,NA,NA,G15,1,gill,⋯,,BFi-13,2026-07-06,1,7/6/26,NA,NA,8.27,3.02,Diluted final sample 1:3
245,Pooled_negative_rack3_2_NA,Pooled,NA,NA,NA,rack,3,2,NA,NA,⋯,,BFi-13,2026-07-08,1,7/10/26,NA,NA,2.20,3.00,Rack 3 negative
246,2024_Warm04_Null_G27_gill,2024,WNA,Warm,4,NA,NA,G27,1,gill,⋯,Redo - clean before indexing,BFi-13,2026-07-31,2,8/4/26,NA,NA,3.76,6.65,
247,negative_control_NA,negative,NANA,NA,NA,NA,NA,NA,NA,NA,⋯,,BFi-13,2026-07-06,1,7/6/26,NA,NA,3.64,3.00,


,Sample_name,Year,Combined_treat,Phase_1_treat,Phase_1_rep,Phase_2_treat,Phase_2_rep,Tag,Phase,Tissue,⋯,Pooled_notes,Index_primers,Index_date,Index_volume,Clean_date,Clean_date_2,Gel_extraction,Final_concentration,X25ng,Notes
,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<int>,<chr>,<int>,<chr>,⋯,<chr>,<chr>,<date>,<int>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>
280,Positive_control_NA,Positive,NANA,NA,NA,NA,NA,NA,NA,NA,⋯,Clean before indexing,BFi-13,2026-07-31,2,NA,NA,8/4/26,5.50,4.55,
281,Pooled_negative_rack3_3_NA,Pooled,NA,NA,NA,rack,3,3,NA,NA,⋯,Clean before indexing,BFi-13,2026-07-31,2,8/4/26,NA,NA,0.26,3.00,Rack 3 negative
282,_,,NANA,NA,NA,NA,NA,NA,NA,,⋯,,,NA,NA,,,,NA,NA,
283,_,,NANA,NA,NA,NA,NA,NA,NA,,⋯,,,NA,NA,,,,NA,NA,


In [35]:
# Remove empty rows 282, 283
meta16S <- meta16S[-282:-283,]
tail(meta16S)

,Sample_name,Year,Combined_treat,Phase_1_treat,Phase_1_rep,Phase_2_treat,Phase_2_rep,Tag,Phase,Tissue,⋯,Pooled_notes,Index_primers,Index_date,Index_volume,Clean_date,Clean_date_2,Gel_extraction,Final_concentration,X25ng,Notes
,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<int>,<chr>,<int>,<chr>,⋯,<chr>,<chr>,<date>,<int>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>
276,2024_Warm05_Cont04_G45_gill,2024,WC,Warm,5,Cont,4,G45,2,gill,⋯,Redo - clean before indexing,BFi-13,2026-07-31,2,NA,NA,8/4/26,4.16,6.01,
277,2024_Both03_Warm04_O41_gill,2024,BW,Both,3,Warm,4,O41,2,gill,⋯,,BFi-13,2026-07-22,2,NA,NA,7/24/26,4.90,5.10,
278,2024_Cont06_Null_W42_gill,2024,CNA,Cont,6,NA,NA,W42,1,gill,⋯,Redo - clean before indexing,BFi-13,2026-07-31,2,8/4/26,NA,NA,2.34,10.68,
279,2024_Hyp02_Hyp01_B3_gut,2024,HH,Hyp,2,Hyp,1,B3,2,gut,⋯,Redo - clean before indexing,BFi-13,2026-07-31,2,NA,NA,8/4/26,2.34,10.68,"Re-index with rack 3 (was rack 1, tube 43)"
280,Positive_control_NA,Positive,NANA,NA,NA,NA,NA,NA,NA,NA,⋯,Clean before indexing,BFi-13,2026-07-31,2,NA,NA,8/4/26,5.50,4.55,
281,Pooled_negative_rack3_3_NA,Pooled,NA,NA,NA,rack,3,3,NA,NA,⋯,Clean before indexing,BFi-13,2026-07-31,2,8/4/26,NA,NA,0.26,3.00,Rack 3 negative


In [36]:
# Save tided csv
write.csv(meta16S, "/project/pi_sarah_gignouxwolfsohn_uml_edu/jade/ORCC_16S/metadata/metadata_samples.csv")

## 10.18.26 | phyloseq on decipher taxonomy

In [ ]:
# Read ASV table
seqtab.nochim <- readRDS("/scratch4/workspace/jade_fiorilla_student_uml_edu-rawseqorcc/dada2/ASV_nochim.rds")

# Read taxonomy table
taxaDecipher <- readRDS("/scratch4/workspace/jade_fiorilla_student_uml_edu-rawseqorcc/dada2/decipher_taxa_sp.rds")

# Read tidied metadata
samples <- read.csv("/project/pi_sarah_gignouxwolfsohn_uml_edu/jade/ORCC_16S/metadata/metadata_samples.csv")

In [ ]:
# Construct phyloseq object
psObject <- phyloseq(otutable(seqtab.nochim, taxa_are_rows = FALSE) ,
                     sample_data(samples),
                     tax_table(taxaDecipher))

In [ ]:
# Save ASV sequences but rename with short names for downstream analysis
dna <- Biostrings::DNAStringSet(taxa_names(ps))
names(dna) <- taxa_names(ps)
ps <- merge_phyloseq(ps, dna)
taxa_names(ps) <- paste0("ASV", seq(ntaxa(ps)))
ps

In [ ]:
plot_richness(ps, x="Combined_treat", measures=c("Shannon", "Simpson"), color="Phase")